# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shiva-sn/ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane:** Structured Content Archetype Clustering

This notebook defines the data contract for the content-level clustering analysis. It uses a local anonymized CSV when available and falls back to the W05 modeling output. The notebook is self-contained and does not require a secret, external warehouse connection, or private query data.

## 1. Unit of analysis + time window

**Unit of analysis:** one row represents one content item for the available analysis snapshot. The downstream W05 model aggregates daily performance to one row per content item and combines it with content attributes and 90-day search/query summaries.

**Time window:** the performance component is a rolling 90-day window ending on the latest available performance date. Content age and days since update are measured relative to that same snapshot.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_CANDIDATES = [
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../outputs/content_archetypes_clustered.parquet"),
    Path("../outputs/content_level_model_dataset.parquet"),
]

DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("No supported data file found. Keep the anonymized CSV in data/raw/ or run W05 first.")

if DATA_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(DATA_PATH)
else:
    df = pd.read_parquet(DATA_PATH)

print("Loaded:", DATA_PATH.resolve())
print("Shape:", df.shape)
print("Columns:", len(df.columns))

assert len(df) > 0
assert "content_id" in df.columns or "content_hash_id" in df.columns
print("Unit-of-analysis dataset loaded successfully.")

## 2. Fields: feature / label / context / excluded

This project is **unsupervised clustering**, so there is no prediction label. The core features are the eight numeric signals used by W05. Identifiers are context only. Query breadth, provider/model metadata, and future/trend-style fields are excluded from the clustering vector.

In [ ]:
core_features = [
    "search_volume", "word_count",
    "content_age_days", "days_since_update",
    "impressions_90d", "ctr_90d",
    "avg_position_90d", "engagement_rate",
]

field_contract = pd.DataFrame([
    ("search_volume", "feature", "Topic-level search demand signal", "median", "snapshot"),
    ("word_count", "feature", "Content length in words", "median", "snapshot"),
    ("content_age_days", "feature", "Age at analysis snapshot", "median", "snapshot"),
    ("days_since_update", "feature", "Days since recorded update", "median", "snapshot"),
    ("impressions_90d", "feature", "Observed search impressions over 90 days", "median + log1p", "90-day window"),
    ("ctr_90d", "feature", "Observed click-through rate over 90 days", "median", "90-day window"),
    ("avg_position_90d", "feature", "Observed average position over 90 days; zero means no position data", "0→NaN, then median", "90-day window"),
    ("engagement_rate", "feature", "Observed engaged-session share", "median", "90-day window"),
    ("client_id / client_hash_id", "context", "Join/group identifier only", "not imputed", "snapshot"),
    ("content_id / content_hash_id", "context", "Content identifier only", "not imputed", "snapshot"),
    ("cluster", "label-like output", "W05-generated cluster assignment, not a source label", "not a feature", "after modeling"),
    ("query breadth", "excluded", "Can encode data-coverage differences rather than substantive archetypes", "excluded", "90-day window"),
    ("future / trend fields", "excluded", "Could introduce later information into a snapshot-defined cluster", "excluded", "future"),
    ("provider/model metadata", "excluded", "Not part of the content-performance archetype definition", "excluded", "snapshot"),
], columns=["field", "bucket", "meaning", "missing_treatment", "availability"])

display(field_contract)

## 3. Verify it with queries (grain, counts, missing values, windows)

The checks below verify the grain, identify duplicate content rows, inspect numeric missingness, and confirm the intended 90-day fields are present. The CSV may already be pre-aggregated, so the exact date window is verified from available columns when possible rather than assumed.

In [ ]:
id_col = "content_id" if "content_id" in df.columns else "content_hash_id"
client_col = "client_id" if "client_id" in df.columns else ("client_hash_id" if "client_hash_id" in df.columns else None)

print("Rows:", len(df))
print("Unique content IDs:", df[id_col].nunique())
print("Duplicate content IDs:", int(df[id_col].duplicated().sum()))
if client_col:
    print("Unique clients:", df[client_col].nunique())

missing_core = (df[core_features].isna().sum().sort_values(ascending=False).to_frame("missing_n"))
missing_core["missing_pct"] = (missing_core["missing_n"] / len(df) * 100).round(2)
display(missing_core)

for candidate in ["report_date", "window_start", "window_end"]:
    if candidate in df.columns:
        s = pd.to_datetime(df[candidate], errors="coerce")
        print(f"{candidate}:", s.min(), "to", s.max())

if "avg_position" in df.columns:
    print("Raw avg_position=0 rows:", int((df["avg_position"] == 0).sum()))
if "avg_position_90d" in df.columns:
    print("W05 avg_position_90d=0 rows:", int((df["avg_position_90d"] == 0).sum()))

## 4. Data limits

This contract supports descriptive clustering only. The dataset can show observed differences among content items, but it cannot establish that a content change caused a later traffic/ranking outcome. A 90-day performance window is a snapshot of observed behavior, not a complete representation of every content lifecycle. Missing data can affect the feature matrix, and a small or rare client/content group may not represent the whole portfolio.

For the downstream W05 model, `avg_position = 0` is interpreted as **no position data**, not a genuine rank. Median imputation is applied to the working feature matrix, and client/content identifiers are not clustering features.

In [ ]:
limits = [
    "No causal inference from cluster membership",
    "90-day observation window may not represent the full content lifecycle",
    "Missingness and median imputation can influence cluster boundaries",
    "Grouped client validation tests unseen-client generalization, not future-time generalization",
    "Rare clusters should be interpreted as narrow observed patterns",
    "The structured dataset does not contain article text, so this is metric-based rather than semantic clustering",
]
for i, item in enumerate(limits, 1):
    print(f"{i}. {item}")

# Contract assertions
assert len(core_features) == 8
assert id_col in df.columns
assert df[id_col].nunique() == len(df), "Expected one row per content item in the contracted snapshot."
print("Contract assertions: PASS")

## Self-check

The notebook is complete when the source data loads, the one-row-per-content grain is verified, the core eight features are explicitly defined, excluded fields are documented, and the data limits are stated using careful decision-support language.

In [ ]:
checks = [
    ("Data source loaded", len(df) > 0),
    ("One row per content", df[id_col].nunique() == len(df)),
    ("Exactly eight core features", len(core_features) == 8),
    ("All core features available", all(c in df.columns for c in core_features)),
    ("No identifier in core feature list", not any(c in core_features for c in ["content_id", "content_hash_id", "client_id", "client_hash_id"])),
    ("No future/trend/label feature names in core vector", not any(k in c.lower() for c in core_features for k in ["future", "trend", "label", "target", "outcome"])),
    ("No query-breadth feature in core vector", not any("query" in c.lower() for c in core_features)),
]
check_df = pd.DataFrame(checks, columns=["check", "passed"])
display(check_df)

if not check_df["passed"].all():
    raise AssertionError("One or more W03 data-contract checks failed.")
print("All W03 data-contract checks PASS.")